# 🎭 Deepfake Detection — ViT-B/16 (7-Class → Binary)

A multi-dataset deepfake detector trained on FaceForensics++, Celeb-DF-v2, and StyleGAN3 faces.  
Architecture: Vision Transformer (ViT-B/16) with focal loss, mixup, TTA, and adaptive thresholding.

---
## 📑 Table of Contents
1. [Configuration & Constants](#1-configuration--constants)
2. [Dataset Preparation (3-Dataset Merge)](#2-dataset-preparation)
3. [Transforms & Dataset Class](#3-transforms--dataset-class)
4. [Model, Loss & Training Utilities](#4-model-loss--training-utilities)
5. [Training Loop](#5-training-loop)
6. [Evaluation — Confusion Matrix & ROC](#6-evaluation)
7. [Error Analysis (Hard FP / FN)](#7-error-analysis)
8. [Robustness Testing](#8-robustness-testing)
9. [Test-Set Final Evaluation](#9-test-set-final-evaluation)
10. [Generator-Specific Accuracy](#10-generator-specific-accuracy)
11. [Speed Benchmark](#11-speed-benchmark)
12. [GradCAM Visualization](#12-gradcam-visualization)
13. [Video Inference](#13-video-inference)
14. [Single-Image Inference](#14-single-image-inference)


## 1. Configuration & Constants

In [ ]:
# ============================================================
# GLOBAL CONFIGURATION — edit paths and hyper-params here only
# ============================================================
import os, random, io
import numpy as np
import torch

# ── Paths ────────────────────────────────────────────────────
FF_BASE    = "/kaggle/input/datasets/gradientvoyager/faceforensics-c23-extracted-faces-100k/dataset_processed_split"
SG3_BASE   = "/kaggle/input/datasets/troykueh/real-vs-fake-faces-stylegan3/Real faces"
CELEB_BASE = "/kaggle/input/datasets/pranabr0y/celebdf-v2image-dataset/Celeb_V2"
DATA_PATH  = "/kaggle/input/datasets/anasacademic/deepfake-3datasets/kaggle/working/dataset_merged_split"
OUTPUT_BASE= "/kaggle/working/dataset_merged_split"
CKPT_DIR   = "/kaggle/working"
BEST_CKPT  = "/kaggle/input/datasets/anasacademic/best-model-epoch13/latest_best_vit_7cls_epoch8.pth"

# ── Classes ──────────────────────────────────────────────────
CATEGORIES = [
    "Real",              # 0
    "DeepFakeDetection", # 1
    "Deepfakes",         # 2
    "Face2Face",         # 3
    "FaceShifter",       # 4
    "FaceSwap",          # 5
    "NeuralTextures",    # 6
]
BINARY_MAP = {i: (0 if i == 0 else 1) for i in range(len(CATEGORIES))}
NUM_CLASSES = len(CATEGORIES)

# ── Training hyper-params ─────────────────────────────────────
MODEL_CHOICE = "vit"
BATCH_SIZE   = 32
EPOCHS       = 15
DROPOUT      = 0.35
IMG_EXTS     = {".jpg", ".jpeg", ".png", ".webp"}
SPLITS       = ["train", "val", "test"]

# ── Reproducibility ───────────────────────────────────────────
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"Classes: {CATEGORIES}")


## 2. Dataset Preparation

Merges three datasets (FaceForensics++, Celeb-DF-v2, StyleGAN3) into a single
`train / val / test` split (70 / 15 / 15 for unsplit sources).  
Run once — skip if `OUTPUT_BASE` already exists.


In [ ]:
import shutil
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm


# ── Helpers ───────────────────────────────────────────────────
def is_valid_image(path: str) -> bool:
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False


def collect_images(folder: str) -> list[str]:
    paths = []
    if not os.path.exists(folder):
        return paths
    for root, _, files in os.walk(folder):
        for f in files:
            if Path(f).suffix.lower() in IMG_EXTS:
                paths.append(os.path.join(root, f))
    return paths


def split_indices(total: int, ratios=(0.70, 0.15)):
    n_train = int(total * ratios[0])
    n_val   = int(total * ratios[1])
    return n_train, n_val


def copy_with_prefix(src_paths, dst_folder, prefix, existing_names=None):
    """Copy images with a prefix; handles name collisions."""
    if existing_names is None:
        existing_names = set(os.listdir(dst_folder))
    skipped = 0
    for src in tqdm(src_paths, desc=f"{prefix} -> {os.path.basename(dst_folder)}", leave=False):
        if not is_valid_image(src):
            skipped += 1
            continue
        fname     = Path(src).name
        candidate = f"{prefix}_{fname}"
        counter   = 0
        while candidate in existing_names:
            counter  += 1
            candidate = f"{prefix}_{Path(fname).stem}_{counter}{Path(fname).suffix}"
        shutil.copy2(src, os.path.join(dst_folder, candidate))
        existing_names.add(candidate)
    return skipped


# ── Phase 1: Copy FaceForensics++ (already split) ────────────
print("=== PHASE 1: Copying FaceForensics++ ===")
for split in SPLITS:
    split_dir = os.path.join(FF_BASE, split)
    if not os.path.exists(split_dir):
        continue
    for category in os.listdir(split_dir):
        src_cat = os.path.join(split_dir, category)
        dst_cat = os.path.join(OUTPUT_BASE, split, category)
        if not os.path.isdir(src_cat):
            continue
        os.makedirs(dst_cat, exist_ok=True)
        for fname in tqdm(os.listdir(src_cat), desc=f"FF++ {split}/{category}", leave=False):
            shutil.copy2(os.path.join(src_cat, fname), os.path.join(dst_cat, fname))


# ── Phase 2: Split Celeb-DF-v2 reals (70/15/15) ─────────────
print("\n=== PHASE 2: Splitting Celeb-DF-v2 Reals ===")

# Case-insensitive search for Train/real folder
celeb_src = None
for tc in ("train", "Train", "TRAIN"):
    for rc in ("real", "Real", "REAL"):
        candidate = os.path.join(CELEB_BASE, tc, rc)
        if os.path.exists(candidate):
            celeb_src = candidate
            break
    if celeb_src:
        break

if celeb_src is None:
    print(f"ERROR: Could not find Celeb-DF Train/Real folder in {CELEB_BASE}")
else:
    celeb_imgs = collect_images(celeb_src)
    random.shuffle(celeb_imgs)
    n_train, n_val = split_indices(len(celeb_imgs))
    buckets = {
        "train": celeb_imgs[:n_train],
        "val":   celeb_imgs[n_train:n_train + n_val],
        "test":  celeb_imgs[n_train + n_val:],
    }
    print(f"  Found {len(celeb_imgs)} Celeb-DF images")
    for split, paths in buckets.items():
        dst = os.path.join(OUTPUT_BASE, split, "Real")
        os.makedirs(dst, exist_ok=True)
        copy_with_prefix(paths, dst, "celeb")


# ── Phase 3: Split StyleGAN3 reals (70/15/15) ────────────────
print("\n=== PHASE 3: Splitting StyleGAN3 Reals ===")
sg3_imgs = collect_images(SG3_BASE)
random.shuffle(sg3_imgs)
n_train, n_val = split_indices(len(sg3_imgs))
sg3_buckets = {
    "train": sg3_imgs[:n_train],
    "val":   sg3_imgs[n_train:n_train + n_val],
    "test":  sg3_imgs[n_train + n_val:],
}
print(f"  Found {len(sg3_imgs)} StyleGAN3 images")
for split, paths in sg3_buckets.items():
    dst = os.path.join(OUTPUT_BASE, split, "Real")
    os.makedirs(dst, exist_ok=True)
    copy_with_prefix(paths, dst, "sg3")


# ── Summary ───────────────────────────────────────────────────
print("\n=== FINAL DATASET SUMMARY ===")
for split in SPLITS:
    split_dir = os.path.join(OUTPUT_BASE, split)
    if not os.path.exists(split_dir):
        continue
    print(f"\n{split}/")
    for cat in sorted(os.listdir(split_dir)):
        cat_dir = os.path.join(split_dir, cat)
        if os.path.isdir(cat_dir):
            print(f"  {cat:<25} {len(os.listdir(cat_dir)):>6} images")


## 3. Transforms & Dataset Class

- **Train**: RandomResizedCrop, flips, colour jitter, Gaussian blur, JPEG compression simulation, Gaussian noise.
- **Val/Test**: Simple resize + normalise.
- `DeepFakeDataset` returns `(image_tensor, 7-class_label)`.  
  Binary conversion is done at inference via `BINARY_MAP`.


In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from collections import Counter


# ── Custom augmentation transforms ───────────────────────────
class AddGaussianNoise:
    """Additive Gaussian noise applied in tensor space."""
    def __init__(self, mean: float = 0.0, std: float = 0.01):
        self.mean = mean
        self.std  = std

    def __call__(self, tensor):
        return tensor + torch.randn(tensor.size()) * self.std + self.mean


class RandomJPEGCompression:
    """Simulate JPEG compression artefacts during training."""
    def __init__(self, quality_range=(60, 90), p: float = 0.3):
        self.quality_range = quality_range
        self.p = p

    def __call__(self, img):
        if random.random() > self.p:
            return img
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=random.randint(*self.quality_range))
        buf.seek(0)
        return Image.open(buf).convert("RGB")


# ── Transform factories ───────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_transforms(train: bool = True):
    if train:
        return transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
            transforms.RandomApply([transforms.GaussianBlur(3)], p=0.2),
            RandomJPEGCompression((60, 90), p=0.3),
            transforms.ToTensor(),
            AddGaussianNoise(0.0, 0.01),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


def get_tta_transforms() -> list:
    """Five TTA variants used at inference time."""
    norm = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
    return [
        transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), norm]),
        transforms.Compose([transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(), norm]),
        transforms.Compose([transforms.Resize((256, 256)), transforms.CenterCrop(224), transforms.ToTensor(), norm]),
        transforms.Compose([transforms.Resize((224, 224)), transforms.ColorJitter(brightness=0.1, contrast=0.1), transforms.ToTensor(), norm]),
        transforms.Compose([transforms.Resize((232, 232)), transforms.CenterCrop(224), transforms.ToTensor(), norm]),
    ]


# ── Dataset ───────────────────────────────────────────────────
class DeepFakeDataset(Dataset):
    """
    Returns (image_tensor, 7-class_label).
    Use BINARY_MAP externally to convert to binary predictions.
    """
    def __init__(self, root_dir: str, transform=None, verbose: bool = True):
        self.transform = transform
        self.samples: list[tuple[str, int]] = []

        for class_idx, category in enumerate(CATEGORIES):
            folder = os.path.join(root_dir, category)
            if not os.path.exists(folder):
                if verbose:
                    print(f"  [WARNING] Missing: {folder}")
                continue
            for img_name in os.listdir(folder):
                self.samples.append((os.path.join(folder, img_name), class_idx))

        if verbose:
            counts = Counter(lbl for _, lbl in self.samples)
            print(f"Loaded {len(self.samples):,} samples from {root_dir}")
            for idx, cat in enumerate(CATEGORIES):
                print(f"  [{idx}] {cat:<25} {counts.get(idx, 0):>6}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label


# ── DataLoaders ───────────────────────────────────────────────
train_dataset = DeepFakeDataset(os.path.join(DATA_PATH, "train"), get_transforms(True))
val_dataset   = DeepFakeDataset(os.path.join(DATA_PATH, "val"),   get_transforms(False))

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Quick sanity check
imgs, lbls = next(iter(train_loader))
print(f"\nBatch shape  : {imgs.shape}")
print(f"7-class labels: {lbls[:8].tolist()}")
print(f"Binary labels : {[BINARY_MAP[l.item()] for l in lbls[:8]]}")


## 4. Model, Loss & Training Utilities

| Component | Details |
|---|---|
| Backbone | ViT-B/16 (pretrained ImageNet) |
| Head | LayerNorm → Linear(768→512) → GELU → Dropout → Linear(512→7) |
| Loss | 7-class Focal Loss with per-class alpha weights |
| Mixup | α = 0.2, applied 20 % of batches |
| Threshold | Grid-searched on val set for best macro-F1 |
| Early stopping | patience = 5, min_delta = 0.002 |


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
from sklearn.metrics import f1_score, classification_report
from tqdm.auto import tqdm


# ── Focal Loss ────────────────────────────────────────────────
class FocalLoss(nn.Module):
    """
    7-class focal loss with per-class alpha weights.
    Default alpha upweights Real slightly to handle class imbalance.
    """
    def __init__(self, alpha=None, gamma: float = 2.0):
        super().__init__()
        if alpha is None:
            alpha = torch.tensor([0.20, 0.13, 0.13, 0.13, 0.14, 0.13, 0.14])
        self.register_buffer("alpha", alpha)
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, reduction="none")
        pt = torch.exp(-ce)
        at = self.alpha[targets]
        return (at * (1 - pt) ** self.gamma * ce).mean()


# ── Model builder ─────────────────────────────────────────────
def build_vit(num_classes: int = NUM_CLASSES, dropout: float = DROPOUT):
    model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
    for p in model.parameters():          # freeze backbone
        p.requires_grad = False
    in_f = model.heads.head.in_features
    model.heads.head = nn.Sequential(
        nn.LayerNorm(in_f),
        nn.Linear(in_f, 512),
        nn.GELU(),
        nn.Dropout(dropout),
        nn.Linear(512, num_classes),
    )
    base_lr    = 2e-5
    vit_blocks = 4   # top-k ViT blocks to unfreeze in Phase 2
    return model, base_lr, vit_blocks


def build_model(choice: str = MODEL_CHOICE, **kwargs):
    if choice == "vit":
        return build_vit(**kwargs)
    raise ValueError(f"Unknown model: {choice}")


# ── Binary helpers ────────────────────────────────────────────
def to_binary(arr, mode: str = "preds"):
    """
    mode='preds' : int tensor/array [N] → binary array
    mode='probs' : float tensor [N,7] → fake probability [N] (sum of fake columns)
    """
    if mode == "preds":
        if isinstance(arr, torch.Tensor):
            arr = arr.cpu().numpy()
        return np.array([BINARY_MAP[int(p)] for p in arr])
    # mode == 'probs'
    if isinstance(arr, torch.Tensor):
        arr = arr.cpu().numpy()
    return arr[:, 1:].sum(axis=1)   # shape [N]


# ── Threshold grid search ─────────────────────────────────────
@torch.no_grad()
def find_best_threshold(model, val_loader, device):
    """Grid-search threshold on val set; returns (best_thresh, fake_probs, binary_labels)."""
    model.eval()
    all_fake_probs, all_bin_labels = [], []

    for images, labels in val_loader:
        probs    = F.softmax(model(images.to(device, non_blocking=True)), dim=1)
        fake_p   = to_binary(probs, mode="probs")
        bin_lbls = to_binary(labels.numpy(), mode="preds")
        all_fake_probs.extend(fake_p.tolist())
        all_bin_labels.extend(bin_lbls.tolist())

    fake_probs  = np.array(all_fake_probs)
    bin_labels  = np.array(all_bin_labels)
    best_thresh, best_f1 = 0.5, 0.0

    for thresh in np.arange(0.30, 0.95, 0.02):
        preds = (fake_probs >= thresh).astype(int)
        f1    = f1_score(bin_labels, preds, average="macro")
        if f1 > best_f1:
            best_f1, best_thresh = f1, thresh

    print(f"  Best threshold: {best_thresh:.2f}  →  Binary Macro-F1: {best_f1:.4f}")
    return best_thresh, fake_probs, bin_labels


# ── Validation ────────────────────────────────────────────────
@torch.no_grad()
def validate(model, val_loader, criterion, device, threshold: float = 0.5):
    """Returns (loss, bin_acc, real_acc, fake_acc, macro_f1, acc_7cls)."""
    model.eval()
    total_loss = 0
    all_7cls_preds, all_bin_preds, all_bin_labels = [], [], []

    for images, labels in val_loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        outputs        = model(images)
        total_loss    += criterion(outputs, labels).item()

        probs      = F.softmax(outputs, dim=1)
        fake_p     = to_binary(probs, mode="probs")
        bin_pred   = (fake_p >= threshold).astype(int)
        bin_labels = to_binary(labels.cpu().numpy(), mode="preds")

        all_7cls_preds.extend(outputs.argmax(1).cpu().numpy())
        all_bin_preds.extend(bin_pred.tolist())
        all_bin_labels.extend(bin_labels.tolist())

    bp = np.array(all_bin_preds)
    bl = np.array(all_bin_labels)
    p7 = np.array(all_7cls_preds)

    # Re-collect ground-truth 7-class labels for accuracy computation
    gt7 = np.concatenate([lbl.numpy() for _, lbl in val_loader])

    bin_acc  = (bp == bl).mean() * 100
    real_acc = (bp[bl == 0] == 0).mean() * 100
    fake_acc = (bp[bl == 1] == 1).mean() * 100
    macro_f1 = f1_score(bl, bp, average="macro")
    acc_7cls = (p7 == gt7[:len(p7)]).mean() * 100

    return total_loss / len(val_loader), bin_acc, real_acc, fake_acc, macro_f1, acc_7cls


# ── Mixup ─────────────────────────────────────────────────────
def mixup_data(x, y, alpha: float = 0.2):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_criterion(criterion, pred, ya, yb, lam):
    return lam * criterion(pred, ya) + (1 - lam) * criterion(pred, yb)


# ── Early stopping ────────────────────────────────────────────
class EarlyStopping:
    def __init__(self, patience: int = 5, min_delta: float = 0.002):
        self.patience    = patience
        self.min_delta   = min_delta
        self.best_f1     = 0.0
        self.counter     = 0
        self.should_stop = False

    def step(self, f1: float) -> bool:
        """Returns True if this is a new best."""
        if f1 > self.best_f1 + self.min_delta:
            self.best_f1 = f1
            self.counter = 0
            return True
        self.counter += 1
        if self.counter >= self.patience:
            self.should_stop = True
        return False


print("All utilities defined.")


## 5. Training Loop

Two-phase fine-tuning:
- **Phase 1** (epochs 0-2): only the head is trained.
- **Phase 2** (epoch 3+): top-4 ViT encoder blocks + LayerNorm are unfrozen.

Checkpoints are saved every epoch; best model (by macro-F1) is also tracked separately.


In [ ]:

import shutil
import os

# Create your checkpoint directory in the writable working folder
CKPT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

# Define paths
original_ckpt_path = "/kaggle/input/datasets/anasacademic/best-model-epoch13/latest_best_vit_7cls_epoch8.pth"
# Assuming MODEL_CHOICE is "vit", the code expects "latest_vit_7cls.pth"
expected_ckpt_path = os.path.join(CKPT_DIR, f"latest_{MODEL_CHOICE}_7cls.pth")

# Copy and rename the file so your script finds it
if os.path.exists(original_ckpt_path):
    shutil.copy(original_ckpt_path, expected_ckpt_path)
    print(f"Copied checkpoint to {expected_ckpt_path}")
else:
    print("Warning: Original checkpoint not found. Check the path.")

In [ ]:
# to train from a certain epoch or from the start
def train(
    model,
    train_loader,
    val_loader,
    device,
    model_name: str,
    base_lr: float,
    vit_blocks: int,
    epochs: int = EPOCHS,
    checkpoint_dir: str = CKPT_DIR,
):
    alpha     = torch.tensor([0.20, 0.13, 0.13, 0.13, 0.14, 0.13, 0.14])
    criterion = FocalLoss(alpha=alpha, gamma=2.0).to(device)

    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=base_lr * 3, weight_decay=0.01,
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-7,
    )
    early_stop = EarlyStopping(patience=5, min_delta=0.002)

    best_ckpt   = os.path.join(checkpoint_dir, f"best_{model_name}_7cls.pth")
    latest_ckpt = os.path.join(checkpoint_dir, f"latest_{model_name}_7cls.pth")

    start_epoch    = 0
    best_threshold = 0.5
    phase2_started = False

    # ── Resume from checkpoint ────────────────────────────────
    if os.path.exists(latest_ckpt):
        print("Checkpoint found — resuming...")
        ckpt = torch.load(latest_ckpt, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])
        phase2_started = ckpt.get("phase2_started", False)

        if phase2_started:
            print("Resuming Phase 2 (top blocks unfrozen)...")
            _unfreeze_top_blocks(model, vit_blocks)
            optimizer = optim.AdamW(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=base_lr, weight_decay=0.01,
            )
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-7,
            )

        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        start_epoch    = ckpt["epoch"] + 1
        best_threshold = ckpt.get("best_threshold", 0.5)
        print(f"Resuming from epoch {start_epoch}")

    # ── Epoch loop ────────────────────────────────────────────
    for epoch in range(start_epoch, epochs):

        # Phase 2 unlock
        if epoch == 3 and not phase2_started:
            print("\n>>> Phase 2: unfreezing top ViT blocks <<<")
            _unfreeze_top_blocks(model, vit_blocks)
            optimizer = optim.AdamW(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=base_lr, weight_decay=0.01,
            )
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-7,
            )
            phase2_started = True

        model.train()
        total_loss, correct_7cls, total = 0, 0, 0
        lr    = optimizer.param_groups[0]["lr"]
        phase = "Phase 2" if phase2_started else "Phase 1"
        print(f"\n--- Epoch {epoch+1}/{epochs} | {phase} | LR: {lr:.2e} ---")

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        for images, labels in pbar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            if random.random() < 0.2:
                images, la, lb, lam = mixup_data(images, labels)
                outputs = model(images)
                loss    = mixup_criterion(criterion, outputs, la, lb, lam)
            else:
                outputs = model(images)
                loss    = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss   += loss.item()
            preds_7       = outputs.argmax(1)
            correct_7cls += preds_7.eq(labels).sum().item()
            total        += labels.size(0)

            bin_pred  = to_binary(preds_7, mode="preds")
            bin_label = to_binary(labels.cpu().numpy(), mode="preds")
            bin_acc   = (bin_pred == bin_label).mean() * 100

            pbar.set_postfix(
                loss=f"{loss.item():.4f}",
                acc7=f"{100.*correct_7cls/total:.1f}%",
                binAcc=f"{bin_acc:.1f}%",
            )

        train_loss = total_loss / len(train_loader)
        train_acc7 = 100.0 * correct_7cls / total

        best_threshold, _, _ = find_best_threshold(model, val_loader, device)
        val_loss, val_acc, real_acc, fake_acc, macro_f1, acc_7val = validate(
            model, val_loader, criterion, device, threshold=best_threshold
        )
        scheduler.step(macro_f1)

        print(
            f"Train Loss: {train_loss:.4f}  |  Val Loss: {val_loss:.4f}\n"
            f"Train 7cls Acc: {train_acc7:.2f}%  |  Val 7cls Acc: {acc_7val:.2f}%\n"
            f"Binary → Val Acc: {val_acc:.2f}%  |  Real: {real_acc:.2f}%  |  "
            f"Fake: {fake_acc:.2f}%  |  Macro-F1: {macro_f1:.4f}"
        )

        # Save latest and per-epoch checkpoints
        ckpt_data = dict(
            epoch=epoch,
            model_state_dict=model.state_dict(),
            optimizer_state_dict=optimizer.state_dict(),
            scheduler_state_dict=scheduler.state_dict(),
            best_threshold=best_threshold,
            phase2_started=phase2_started,
        )
        torch.save(ckpt_data, latest_ckpt)
        torch.save(
            {"model_state_dict": model.state_dict(), "best_threshold": best_threshold},
            os.path.join(checkpoint_dir, f"model_epoch_{epoch+1}.pth"),
        )

        if early_stop.step(macro_f1):
            torch.save(
                {"model_state_dict": model.state_dict(), "best_threshold": best_threshold},
                best_ckpt,
            )
            print(f"  Best model saved  (Macro-F1: {macro_f1:.4f}, threshold: {best_threshold:.2f})")

        if early_stop.should_stop:
            print(f"\nEarly stopping at epoch {epoch+1}.")
            break

    # ── Final TTA evaluation ──────────────────────────────────
    print("\n=== FINAL EVALUATION (TTA on val set) ===")
    ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    best_threshold = ckpt["best_threshold"]
    model.eval()

    val_raw  = DeepFakeDataset(os.path.join(DATA_PATH, "val"), transform=None, verbose=False)
    tta_tfms = get_tta_transforms()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for idx in tqdm(range(len(val_raw)), desc="TTA inference"):
            img_pil, lbl7 = val_raw[idx]
            bin_label = BINARY_MAP[lbl7]
            avg_fake  = np.mean([
                F.softmax(model(tfm(img_pil).unsqueeze(0).to(device)), dim=1)[0][1:].sum().item()
                for tfm in tta_tfms
            ])
            all_preds.append(int(avg_fake >= best_threshold))
            all_labels.append(bin_label)

    print(classification_report(all_labels, all_preds, target_names=["Real", "Fake"]))
    print(f"Optimal threshold: {best_threshold:.2f}")
    return best_threshold


def _unfreeze_top_blocks(model, vit_blocks: int):
    for p in model.encoder.layers[-vit_blocks:].parameters():
        p.requires_grad = True
    for p in model.encoder.ln.parameters():
        p.requires_grad = True


# ── Build & train ─────────────────────────────────────────────
model, base_lr, vit_blocks = build_model(MODEL_CHOICE)
model = model.to(DEVICE)
print("Model ready.")

best_threshold = train(
    model        = model,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = DEVICE,
    model_name   = MODEL_CHOICE,
    base_lr      = base_lr,
    vit_blocks   = vit_blocks,
)


## 6. Evaluation — Confusion Matrix & ROC

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc

# ── Load model for evaluation ─────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, _, _ = build_model(MODEL_CHOICE)
model = model.to(device)

ckpt = torch.load(BEST_CKPT, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
best_threshold = ckpt["best_threshold"]
model.eval()
print(f"Loaded model  |  threshold: {best_threshold:.2f}")

# ── TTA inference on val set ──────────────────────────────────
val_raw  = DeepFakeDataset(os.path.join(DATA_PATH, "val"), transform=None, verbose=False)
tta_tfms = get_tta_transforms()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for idx in tqdm(range(len(val_raw)), desc="Val TTA"):
        img_pil, lbl7 = val_raw[idx]
        fake_p = np.mean([
            F.softmax(model(tfm(img_pil).unsqueeze(0).to(device)), dim=1)[0][1:].sum().item()
            for tfm in tta_tfms
        ])
        all_probs.append(fake_p)
        all_preds.append(int(fake_p >= best_threshold))
        all_labels.append(BINARY_MAP[lbl7])

# ── Confusion matrix ──────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Pred Real", "Pred Fake"],
            yticklabels=["Actual Real", "Actual Fake"])
axes[0].set_title("Binary Confusion Matrix")
axes[0].set_ylabel("True Label")
axes[0].set_xlabel("Predicted Label")

# ── ROC curve ─────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(all_labels, all_probs)
roc_auc     = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color="darkorange", lw=2, label=f"AUC = {roc_auc:.3f}")
axes[1].plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--", label="Random")
axes[1].set(xlim=[0, 1], ylim=[0, 1.05],
            xlabel="False Positive Rate", ylabel="True Positive Rate",
            title="ROC Curve")
axes[1].legend(loc="lower right")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Val AUC: {roc_auc:.4f}")


## 7. Robustness Testing
Evaluate accuracy under JPEG compression, Gaussian blur, and camera noise.

In [ ]:
from PIL import ImageFilter
from sklearn.metrics import accuracy_score

# ── Degradation functions ─────────────────────────────────────
def apply_jpeg(img, quality: int = 30):
    """Simulate social-media compression."""
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    return Image.open(buf).convert("RGB")

def apply_blur(img, radius: int = 3):
    """Simulate out-of-focus blur."""
    return img.filter(ImageFilter.GaussianBlur(radius=radius))

def apply_noise(img, severity: float = 0.15):
    """Simulate low-light camera noise."""
    t     = transforms.ToTensor()(img)
    t     = torch.clamp(t + torch.randn(t.size()) * severity, 0, 1)
    return transforms.ToPILImage()(t)

conditions = {
    "Baseline (Clean)":    lambda x: x,
    "Heavy JPEG (Q=30)":   lambda x: apply_jpeg(x, 30),
    "Gaussian Blur (R=3)": lambda x: apply_blur(x, 3),
    "Camera Noise":        lambda x: apply_noise(x, 0.15),
}

base_tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Sample 1000 images for speed
np.random.seed(42)
subset_idx = np.random.choice(len(val_raw), min(1000, len(val_raw)), replace=False)

results = {}
model.eval()

with torch.no_grad():
    for cond_name, degrade in conditions.items():
        preds, labels = [], []
        for idx in tqdm(subset_idx, desc=cond_name):
            img_pil, lbl7 = val_raw[idx]
            tensor    = base_tfm(degrade(img_pil)).unsqueeze(0).to(device)
            probs     = F.softmax(model(tensor), dim=1)[0]
            fake_prob = probs[1:].sum().item()
            preds.append(int(fake_prob >= best_threshold))
            labels.append(BINARY_MAP[lbl7])
        results[cond_name] = accuracy_score(labels, preds)

print("\n" + "=" * 45)
print("ROBUSTNESS RESULTS")
print("=" * 45)
print(f"{'Condition':<25} | Accuracy")
print("-" * 45)
for cond, acc in results.items():
    print(f"{cond:<25} | {acc*100:.2f}%")
print("=" * 45)


## 8. Test-Set Final Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

test_dir = os.path.join(DATA_PATH, "test")
if not os.path.exists(test_dir):
    print(f"Test folder not found: {test_dir}")
else:
    test_raw    = DeepFakeDataset(test_dir, transform=None, verbose=False)
    tta_tfms    = get_tta_transforms()
    test_preds, test_labels, test_probs = [], [], []

    model.eval()
    with torch.no_grad():
        for idx in tqdm(range(len(test_raw)), desc="Test TTA"):
            img_pil, lbl7 = test_raw[idx]
            fake_p = np.mean([
                F.softmax(model(tfm(img_pil).unsqueeze(0).to(device)), dim=1)[0][1:].sum().item()
                for tfm in tta_tfms
            ])
            test_probs.append(fake_p)
            test_preds.append(int(fake_p >= best_threshold))
            test_labels.append(BINARY_MAP[lbl7])

    acc = accuracy_score(test_labels, test_preds)
    print("\n" + "=" * 50)
    print("FINAL TEST SET PERFORMANCE")
    print("=" * 50)
    print(f"Overall Accuracy: {acc*100:.2f}%\n")
    print(classification_report(test_labels, test_preds, target_names=["Real", "Fake"], digits=4))

    cm_test = confusion_matrix(test_labels, test_preds)
    tn, fp, fn, tp = cm_test.ravel()
    print(f"TN (Real→Real): {tn}  |  FP (Real→Fake): {fp}")
    print(f"FN (Fake→Real): {fn}  |  TP (Fake→Fake): {tp}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc

# Calcul des taux de faux positifs (fpr) et vrais positifs (tpr) pour la courbe ROC
fpr, tpr, thresholds = roc_curve(test_labels, test_probs)
roc_auc = auc(fpr, tpr)

# Création d'une figure unique avec 1 ligne et 2 colonnes (côte à côte)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- 1. AFFICHAGE DE LA MATRICE DE CONFUSION (à gauche) ---
sns.heatmap(
    cm_test, 
    annot=True, 
    fmt="d", 
    cmap="Blues", 
    xticklabels=["Real", "Fake"], 
    yticklabels=["Real", "Fake"],
    ax=axes[0]  # Indique d'afficher sur le premier graphique
)
axes[0].set_title("Matrice de Confusion (Test TTA)")
axes[0].set_ylabel("Vrais Labels")
axes[0].set_xlabel("Prédictions")

# --- 2. AFFICHAGE DE LA COURBE ROC (à droite) ---
axes[1].plot(fpr, tpr, color="darkorange", lw=2, label=f"Courbe ROC (AUC = {roc_auc:.4f})")
axes[1].plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")  # Ligne aléatoire
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel("Taux de Faux Positifs (FPR)")
axes[1].set_ylabel("Taux de Vrais Positifs (TPR)")
axes[1].set_title("Courbe ROC - Détection de DeepFakes")
axes[1].legend(loc="lower right")
axes[1].grid(True, linestyle="--", alpha=0.6)

# Ajustement automatique de l'espacement et affichage
plt.tight_layout()
plt.show()


## 9. Generator-Specific Accuracy
Per-class breakdown to identify which deepfake generator is hardest to detect.

In [ ]:
from torch.utils.data import DataLoader

val_tfm     = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_dataset2 = DeepFakeDataset(os.path.join(DATA_PATH, "val"), transform=val_tfm, verbose=False)
val_loader2  = DataLoader(val_dataset2, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

class_correct = {cat: 0 for cat in CATEGORIES}
class_total   = {cat: 0 for cat in CATEGORIES}

model.eval()
with torch.no_grad():
    for images, labels in tqdm(val_loader2, desc="Generator accuracy"):
        images = images.to(device, non_blocking=True)
        probs  = F.softmax(model(images), dim=1)

        for i in range(len(labels)):
            cat       = CATEGORIES[labels[i].item()]
            fake_prob = probs[i, 1:].sum().item()
            pred_fake = 1 if fake_prob >= best_threshold else 0
            actual    = 0 if labels[i].item() == 0 else 1
            class_correct[cat] += int(pred_fake == actual)
            class_total[cat]   += 1

print("\n" + "=" * 50)
print("GENERATOR-SPECIFIC ACCURACY")
print("=" * 50)
for cat in CATEGORIES:
    total = class_total[cat]
    if total > 0:
        acc = class_correct[cat] / total * 100
        print(f"{cat:<22}: {acc:>6.2f}%  ({class_correct[cat]}/{total})")
    else:
        print(f"{cat:<22}: N/A")
print("=" * 50)


## 10. Speed Benchmark

In [ ]:
import time

def benchmark(model, device, num_runs: int = 100):
    model.to(device).eval()
    dummy = torch.randn(1, 3, 224, 224).to(device)

    # Warm-up
    with torch.no_grad():
        for _ in range(10):
            model(dummy)

    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(num_runs):
            model(dummy)
    elapsed = time.perf_counter() - start

    latency_ms = elapsed / num_runs * 1000
    fps        = num_runs / elapsed

    print("\n" + "=" * 40)
    print("SPEED BENCHMARK")
    print("=" * 40)
    print(f"Hardware  : {str(device).upper()}")
    print(f"Model     : ViT-B/16")
    print(f"Latency   : {latency_ms:.2f} ms / frame")
    print(f"Throughput: {fps:.1f} FPS")
    print("=" * 40)
    if fps >= 30:
        print("✅ Real-time capable (≥ 30 FPS)")
    elif fps >= 10:
        print("⚠️  Near real-time — use frame skipping")
    else:
        print("🔴 Offline batch processing recommended")

benchmark(model, DEVICE)


## 11. GradCAM Visualization
Visualise which facial regions the model attends to when making a prediction.

In [ ]:
!pip install grad-cam


In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

# ── Reload model on CPU for GradCAM ──────────────────────────
cam_device = torch.device("cpu")
cam_model  = build_model(MODEL_CHOICE)[0]
cam_model.load_state_dict(
    torch.load(BEST_CKPT, map_location=cam_device, weights_only=False)["model_state_dict"]
)
cam_model.eval()

for param in cam_model.parameters():
    param.requires_grad = True

def reshape_transform(tensor, height: int = 14, width: int = 14):
    """Reshape ViT patch tokens [B, 197, 768] → [B, 768, 14, 14]."""
    result = tensor[:, 1:, :]   # drop class token
    result = result.reshape(tensor.size(0), height, width, tensor.size(2))
    return result.transpose(2, 3).transpose(1, 2)

target_layers = [cam_model.encoder.layers[-1].ln_1]
cam           = GradCAM(model=cam_model, target_layers=target_layers,
                         reshape_transform=reshape_transform)

def visualize_gradcam(image_path: str):
    img_pil    = Image.open(image_path).convert("RGB").resize((224, 224))
    img_float  = np.float32(img_pil) / 255.0
    tfm        = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    input_tensor  = tfm(img_pil).unsqueeze(0)
    grayscale_cam = cam(input_tensor=input_tensor, targets=None)[0]
    overlay       = show_cam_on_image(img_float, grayscale_cam, use_rgb=True)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img_pil);  axes[0].set_title("Original");      axes[0].axis("off")
    axes[1].imshow(overlay);  axes[1].set_title("GradCAM Heatmap"); axes[1].axis("off")
    plt.tight_layout()
    plt.show()

# Example — replace with your own image path
GRADCAM_IMAGE = "/kaggle/input/datasets/gradientvoyager/faceforensics-c23-extracted-faces-100k/dataset_processed_split/test/Real/071_f311.jpg"
visualize_gradcam(GRADCAM_IMAGE)


## 14. Single-Image Inference

In [ ]:
import matplotlib.pyplot as plt

def predict_image(image_path: str, threshold: float = 0.5):
    """Classify a single image, crop the face if needed, and display it with the prediction."""
    if not os.path.exists(image_path):
        print(f"Error: file not found — {image_path}")
        return

    img_pil = Image.open(image_path).convert("RGB")
    w, h = img_pil.size
    
    # Copie de sécurité pour l'affichage final
    img_display = img_pil.copy()
    
    # ── AJOUT : Même logique exacte de détection que votre fonction vidéo ──
    try:
        img_np = np.array(img_pil)
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
        
        if len(faces) > 0:
            # Récupération stricte du premier visage comme dans la vidéo [0]
            x, y, face_w, face_h = faces[0]  
            
            x_min = max(0, int(x))
            y_min = max(0, int(y))
            x_max = min(w, int(x + face_w))
            y_max = min(h, int(y + face_h))
            
            # Recadrage pour l'évaluation par le ViT
            img_pil = img_pil.crop((x_min, y_min, x_max, y_max))
            
    except Exception as e:
        print(f"Vision preprocessing warning: {e}. Using full image instead.")
    # ──────────────────────────────────────────────────────────────────────────────────

    model.eval()
    
    # Récupération dynamique du device (exactement comme votre fonction vidéo)
    model_device = next(model.parameters()).device

    with torch.no_grad():
        # Utilisation de inference_tfm (comme dans votre fonction vidéo)
        tensor    = inference_tfm(img_pil).unsqueeze(0).to(model_device)
        probs_7   = F.softmax(model(tensor), dim=1)[0]
        fake_prob = probs_7[1:].sum().item()

    verdict = "FAKE" if fake_prob >= threshold else "REAL"

    plt.figure(figsize=(6, 6))
    plt.imshow(img_display) # Affichage de l'image originale propre sans carré rouge
    
    plt.title(f"Prediction: {verdict}\nFake Probability: {fake_prob*100:.2f}%",
              fontsize=14, fontweight="bold")
    plt.axis("off")
    plt.show()

# Exemple d'exécution
IMAGE_PATH = "/kaggle/input/datasets/anasacademic/test-still/still_test/real/not_cropped1.jpg"
predict_image(IMAGE_PATH, threshold=0.5)


## 13. Video Inference
Face detection (Haar cascade) + per-frame deepfake probability, aggregated to a single verdict.

In [ ]:
import cv2
import os
import torch
import torch.nn.functional as F
from PIL import Image
import numpy as np

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

inference_tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def predict_video(video_path: str, frame_interval: int = 10, threshold: float = 0.5): # fallback threshold
    """
    Analyse a video file frame by frame, crop the largest detected face,
    and return a REAL / FAKE verdict with an average fake probability.
    """
    if not os.path.exists(video_path):
        print(f"Error: file not found — {video_path}")
        return

    # ---> NEW: Dynamically grab whatever device the model is currently on
    current_device = next(model.parameters()).device

    cap           = cv2.VideoCapture(video_path)
    fake_probs    = []
    frame_count   = 0
    analyzed      = 0
    model.eval()

    with torch.no_grad():
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            if frame_count % frame_interval == 0:
                gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))

                if len(faces) > 0:
                    x, y, w, h = faces[0]
                    m  = int(w * 0.1)   # margin
                    y1, y2 = max(0, y - m), min(frame.shape[0], y + h + m)
                    x1, x2 = max(0, x - m), min(frame.shape[1], x + w + m)
                    face   = Image.fromarray(cv2.cvtColor(frame[y1:y2, x1:x2], cv2.COLOR_BGR2RGB))

                    # ---> CHANGED: Use current_device instead of the global device variable
                    input_tensor = inference_tfm(face).unsqueeze(0).to(current_device)
                    probs_7      = F.softmax(model(input_tensor), dim=1)[0]
                    
                    fake_probs.append(probs_7[1:].sum().item())
                    analyzed += 1

            frame_count += 1

    cap.release()

    if analyzed == 0:
        print("No faces detected in video.")
        return

    avg_p   = np.mean(fake_probs)
    verdict = "FAKE 🔴" if avg_p >= threshold else "REAL 🟢"
    print("\n" + "=" * 50)
    print("VIDEO EVALUATION REPORT")
    print("=" * 50)
    print(f"File            : {os.path.basename(video_path)}")
    print(f"Frames analyzed : {analyzed}  (face found)")
    print(f"Avg fake prob   : {avg_p*100:.2f}%")
    print(f"Verdict         : {verdict}")
    print("=" * 50)


# Example
# Make sure 'best_threshold' is defined in your notebook, or pass a float directly.
VIDEO_PATH = "/kaggle/input/datasets/anasacademic/test-data-frommerged/deepfake14.mp4"
predict_video(VIDEO_PATH, frame_interval=10, threshold=0.5)

In [ ]:
sum(p.numel() for p in model.parameters())/1e6